# Rubisco cDNA Analysis
## *Arabidopsis thaliana* — AT3G07670

**Accession:** NM_111646.3  
**Gene:** Rubisco methyltransferase family protein  
**Organism:** *Arabidopsis thaliana* (thale cress)

---

This notebook walks through a complete bioinformatics analysis of a plant cDNA sequence, covering four tasks:

| Task | Description |
|------|-------------|
| 1 | Read and parse a FASTA file |
| 2 | Calculate GC content |
| 3 | Extract the Coding DNA Sequence (CDS) |
| 4 | Translate the CDS into a protein sequence |

> **Requirements:** Python 3 standard library only — no external packages needed.

---
## Setup: Codon Table

Before we touch any sequence data, we define the **codon table** — the dictionary that maps every possible DNA triplet (codon) to its corresponding amino acid.

### Background
- DNA is read in groups of **3 nucleotides** called **codons**.
- There are **4³ = 64** possible codons.
- They encode **20 amino acids** plus **3 stop signals**.
- This mapping is called the **genetic code** and is nearly universal across all life.

In Python we represent the genetic code as a `dict` where:
- **key** = 3-letter codon string (e.g. `'ATG'`)
- **value** = single-letter amino acid code (e.g. `'M'`) or `'STOP'`

The start codon `ATG` encodes **Methionine (M)** and also signals the ribosome to begin translation.

In [1]:
# ============================================================
# Bioinformatics Assignment: Rubisco cDNA Analysis
# Gene: AT3G07670 (Arabidopsis thaliana)
# Accession: NM_111646.3
# ============================================================

# Codon table dictionary
# Maps every triplet codon to its single-letter amino acid code.
# STOP codons are represented as 'STOP'.
codon_table = {
    # Phenylalanine (F)
    'TTT': 'F', 'TTC': 'F',
    # Leucine (L)
    'TTA': 'L', 'TTG': 'L', 'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    # Isoleucine (I)
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I',
    # Methionine / Start (M)
    'ATG': 'M',
    # Valine (V)
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    # Serine (S)
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S', 'AGT': 'S', 'AGC': 'S',
    # Proline (P)
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    # Threonine (T)
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    # Alanine (A)
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    # Tyrosine (Y)
    'TAT': 'Y', 'TAC': 'Y',
    # STOP codons
    'TAA': 'STOP', 'TAG': 'STOP', 'TGA': 'STOP',
    # Histidine (H)
    'CAT': 'H', 'CAC': 'H',
    # Glutamine (Q)
    'CAA': 'Q', 'CAG': 'Q',
    # Asparagine (N)
    'AAT': 'N', 'AAC': 'N',
    # Lysine (K)
    'AAA': 'K', 'AAG': 'K',
    # Aspartic acid (D)
    'GAT': 'D', 'GAC': 'D',
    # Glutamic acid (E)
    'GAA': 'E', 'GAG': 'E',
    # Cysteine (C)
    'TGT': 'C', 'TGC': 'C',
    # Tryptophan (W)
    'TGG': 'W',
    # Arginine (R)
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R', 'AGA': 'R', 'AGG': 'R',
    # Glycine (G)
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
}

print(f"Codon table loaded — {len(codon_table)} codons defined.")

Codon table loaded — 64 codons defined.


---
## Task 1: Read and Clean the FASTA Sequence File

### What is FASTA format?

FASTA is the most common plain-text format for storing biological sequences. Every FASTA file has two parts:

```
>NM_111646.3 Arabidopsis thaliana Rubisco ... mRNA   ← header line (starts with >)
TTTAAAAACATTATCTTTTAATTTTGTCATGCCCC...               ← sequence lines
TCGTTTTCTCTTTTCCGATGGCAAAAGCTTGCCTC...
```

### Strategy

1. Open the file with `open()`
2. Loop over every line and call `.strip()` to remove whitespace/newlines
3. Skip any line starting with `>` — that is metadata, not sequence
4. Append each remaining line to a list called `sequence_lines`
5. Join all items in the list into **one unbroken string** using `"".join()`

> **Why `"".join()`?**  
> A FASTA file wraps sequence at ~70 characters per line for readability. Joining with an empty string `""` glues all fragments back into one continuous sequence without adding any separator characters.

In [2]:
# =====================================================================
# Task 1: Read and clean the FASTA sequence file
# =====================================================================
# FASTA is a standard bioinformatics text format.
# The first line (header) starts with '>' and contains metadata.
# All subsequent lines contain the actual nucleotide sequence.
print("=" * 60)
print("Task 1: Reading FASTA File")
print("=" * 60)

# The name of our FASTA file containing the cDNA sequence.
# The notebook lives in notebooks/, so we go one level up with '../'
fasta_filename = "../rubisco.fasta"

# Initialise an empty list to collect each sequence line.
# We will append one line at a time as we read through the file.
sequence_lines = []

# open() takes two arguments: the filename, and the mode ('r' = read).
# Using 'with' ensures the file is automatically closed after reading.
with open(fasta_filename, "r") as file:
    for line in file:
        # .strip() removes invisible whitespace at both ends of the line,
        # including '\n' (newline) characters that Python reads from the file.
        line = line.strip()
        # The header line starts with '>'. We skip it because it is
        # descriptive metadata, not part of the nucleotide sequence.
        if not line.startswith(">"):
            sequence_lines.append(line)

# "".join() concatenates all strings in the list using "" as the separator,
# producing one unbroken string of nucleotide characters.
clean_sequence = "".join(sequence_lines)

# Confirm the file was successfully read, and report the sequence length
print("File successfully read and loaded:", fasta_filename)
print("Total sequence length:           ", len(clean_sequence), "bases")

Task 1: Reading FASTA File
File successfully read and loaded: ../rubisco.fasta
Total sequence length:            2067 bases


> **Note:** The notebook file (`rubisco-cDNA-analysis.ipynb`) lives inside the `notebooks/` folder, while `rubisco.fasta` is in the project root one level up. The path `"../rubisco.fasta"` navigates up one directory to find the file. Always launch Jupyter from the project root (`jupyter notebook`) so relative paths resolve correctly.

---
## Task 2: Calculate GC Content

### What is GC content?

**GC content** is the percentage of nucleotides in a DNA or RNA sequence that are either **Guanine (G)** or **Cytosine (C)**.

$$\text{GC\%} = \frac{G + C}{\text{Total bases}} \times 100$$

### Why does it matter?

- G–C base pairs form **3 hydrogen bonds** (vs 2 for A–T), making GC-rich regions more thermally stable.
- GC content is used to:
  - Compare genomes across species
  - Design PCR primers
  - Predict melting temperatures
  - Identify coding vs non-coding regions

### Python method used

We use the built-in **`.count()`** string method, which returns how many times a character (or substring) appears in a string:

```python
"ATGCGC".count("G")  # returns 2
```

In [3]:
# =====================================================================
# Task 2: Calculate GC content
# =====================================================================
# GC content is the percentage of nucleotides in a DNA sequence that
# are either Guanine (G) or Cytosine (C). It is a fundamental measure
# used to characterise genomes and assess sequence quality.
print()
print("=" * 60)
print("Task 2: GC Content")
print("=" * 60)

# .count() is a built-in string method that returns how many times
# a given character (or substring) appears in the string.
g_count = clean_sequence.count("G")
c_count = clean_sequence.count("C")

# GC % = ((number of G + number of C) / total number of bases) x 100
total_bases = len(clean_sequence)
gc_content = ((g_count + c_count) / total_bases) * 100

print("G count:    ", g_count)
print("C count:    ", c_count)
print("GC Content: {:.2f}%".format(gc_content))


Task 2: GC Content
G count:     423
C count:     431
GC Content: 41.32%


---
## Task 3: Extract the CDS (Coding DNA Sequence)

### What is a CDS?

A messenger RNA (mRNA) molecule is longer than just its protein-coding region. It has:

```
5'--[ 5' UTR ]--[ START codon | CDS | STOP codon ]--[ 3' UTR ]--3'
                  ↑ position 88                  ↑ position 1602
```

- **5' UTR** — untranslated region before the start codon
- **CDS** — the coding sequence, from `ATG` (start) through the stop codon
- **3' UTR** — untranslated region after the stop codon

For our Rubisco gene, the annotation tells us:
- CDS starts at **biological position 88** (1-based)
- CDS ends at **biological position 1602** (1-based, inclusive of stop codon)

### 1-based vs 0-based indexing

Biologists count sequence positions starting from **1**.  
Python string indices start from **0**.

| Biological | Python slice |
|------------|-------------|
| Position 88 (start) | Index `87` |
| Position 1602 (end) | Index `1602` (Python end is exclusive, so this is correct) |

```python
cds_sequence = clean_sequence[87:1602]
#                              ↑    ↑
#                        pos 88    pos 1602 (exclusive upper bound)
```

Expected CDS length: **1602 − 88 + 1 = 1515 bases** (505 codons × 3)

In [4]:
# =====================================================================
# Task 3: Extract the CDS (Coding DNA Sequence)
# =====================================================================
# The CDS is the portion of the mRNA that is actually translated into
# protein. It begins at the start codon (ATG) and ends at a stop codon.
# We isolate it by slicing the full sequence using known coordinates.
print()
print("=" * 60)
print("Task 3: CDS Extraction")
print("=" * 60)

# The biological coordinates are 1-based (position 88 to 1602).
# Python uses 0-based indexing, so position 88 becomes index 87.
# The end index 1602 is exclusive in Python slicing, which is correct here.
cds_sequence = clean_sequence[87:1602]

print("CDS sequence isolated. Length:", len(cds_sequence), "bases")


Task 3: CDS Extraction
CDS sequence isolated. Length: 1515 bases


---
## Task 4: Translate CDS to Protein

### What is translation?

**Translation** is the biological process by which the ribosome reads an mRNA sequence and assembles a chain of amino acids (a protein).

The ribosome reads the CDS **three nucleotides at a time**. Each triplet is called a **codon**, and it maps to exactly one amino acid (or a stop signal) via the genetic code.

```
CDS:     ATG  AAA  GCT  TGC  CTC  TTG  ...  TGA
         ↓    ↓    ↓    ↓    ↓    ↓         ↓
Protein:  M    K    A    C    L    L   ...  STOP
```

### Algorithm

1. Start at index `0` of the CDS
2. Step forward in increments of `3` using `range(0, len(cds_sequence), 3)`
3. Slice out the 3-character codon: `codon = cds_sequence[i:i+3]`
4. Look up the codon in `codon_table` using `.get()`
5. If the result is `'STOP'`, **break** — translation is complete
6. Otherwise, append the amino acid to `protein_sequence`
7. After the loop, join the list into a single string with `"".join()`

### Expected result

- CDS = 1515 bases → 505 codons
- Last codon is a STOP → protein length = **504 amino acids**

In [5]:
# =====================================================================
# Task 4: Translate CDS to protein
# =====================================================================
# Translation converts a nucleotide sequence into an amino acid sequence.
# The ribosome reads codons (groups of 3 nucleotides) from the CDS and
# maps each codon to an amino acid using the genetic code.
print()
print("=" * 60)
print("Task 4: Translation")
print("=" * 60)

protein_sequence = []  # Initiate an empty list to collect amino acids

# range(0, length, 3) generates start positions: 0, 3, 6, 9 ...
# Each iteration extracts one codon of exactly 3 characters.
for i in range(0, len(cds_sequence), 3):
    codon = cds_sequence[i:i + 3]

    # Discard any incomplete codon at the very end of the sequence
    if len(codon) < 3:
        break

    # .get(codon, "?") looks up the codon; returns "?" if not found
    amino_acid = codon_table.get(codon, "?")

    # A STOP codon signals the ribosome to release the protein chain
    if amino_acid == "STOP":
        break

    protein_sequence.append(amino_acid)

# Join the amino acid list into a single protein string
final_protein = "".join(protein_sequence)

print("Final Protein Length:", len(final_protein), "amino acids.")
print("Protein Sequence:")
print(final_protein)
print("=" * 60)


Task 4: Translation
Final Protein Length: 504 amino acids.
Protein Sequence:
MAKACLLQSTLLPAYSPLHKLRNQNITLSFSPLPLSRCRPGIHCSVSAGETTIQSMEEAPKISWGCEIDSLENATSLQNWLSDSGLPPQKMAIDRVDIGERGLVASQNLRKGEKLLFVPPSLVISADSEWTNAEAGEVMKRYDVPDWPLLATYLISEASLQKSSRWFNYISALPRQPYSLLYWTRTELDMYLEASQIRERAIERITNVVGTYEDLRSRIFSKHPQLFPKEVFNDETFKWSFGILFSRLVRLPSMDGRFALVPWADMLNHNCEVETFLDYDKSSKGVVFTTDRPYQPGEQVFISYGNKSNGELLLSYGFVPREGTNPSDSVELALSLRKNDKCYEEKLDALKKHGLSTPQCFPVRITGWPMELMAYAYLVVSPPDMRNNFEEMAKAASNKTSTKNDLKYPEIEEDALQFILDSCETSISKYSRFLKESGSMDLDITSPKQLNRKAFLKQLAVDLSTSERRILYRAQYILRRRLRDIRSGELKALRLFSGLRNFFK


---
## Summary of Results

| Analysis | Result |
|----------|--------|
| Full sequence length | 2067 bases |
| G count | 423 |
| C count | 431 |
| GC content | 41.32% |
| CDS coordinates (1-based) | 88 – 1602 |
| CDS length | 1515 bases |
| Protein length | 504 amino acids |
| Protein (first 10 aa) | `MAKACLLQST...` |

---

## Key Python Concepts Used

| Concept | Used for |
|---------|----------|
| `open()` + `with` | Reading the FASTA file safely |
| `.strip()` | Removing newline characters from each line |
| `.startswith()` | Detecting and skipping the FASTA header |
| `.append()` | Building the list of sequence lines |
| `"".join()` | Merging list items into one continuous string |
| `.count()` | Counting G and C nucleotides |
| String slicing `[87:1602]` | Extracting the CDS region |
| `range(0, n, 3)` | Stepping through the CDS three bases at a time |
| `dict.get()` | Looking up a codon in the translation table |
| `"".join()` | Assembling the final protein string |

---
*Gene: AT3G07670 — Arabidopsis thaliana Rubisco methyltransferase family protein*